# Detecting Sensor Drift & Stuck-At Faults

IoT sensors fail silently. A temperature probe slowly drifts out of calibration, or a humidity sensor locks onto a single value and never changes. Your pipeline keeps ingesting the data, your dashboards look fine, and nobody notices until a warehouse full of product is ruined.

This notebook builds two lightweight detectors using only pandas and numpy:

1. **Drift detection** -- rolling z-scores flag readings that gradually deviate from the fleet baseline
2. **Stuck-at detection** -- zero-variance rolling windows catch sensors frozen on a single value

We simulate a fleet of warehouse sensors with realistic failure modes injected at known timestamps, then evaluate how well each detector catches them.

In [ ]:
# Install dependencies if running in Colab
!pip install pandas numpy matplotlib --quiet

## Generate Synthetic Sensor Fleet

We simulate 10 temperature sensors reporting every 5 minutes over 7 days. Normal readings fluctuate around 22°C with gaussian noise. Two sensors have injected faults:

- **sensor_03**: begins drifting upward at hour 80 (~+0.05°C per reading)
- **sensor_07**: gets stuck at exactly 21.8°C from hour 100 onward

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

N_SENSORS = 10
INTERVAL_MIN = 5
DAYS = 7
READINGS_PER_SENSOR = (DAYS * 24 * 60) // INTERVAL_MIN  # 2016

timestamps = pd.date_range("2024-06-01", periods=READINGS_PER_SENSOR, freq=f"{INTERVAL_MIN}min")

rows = []
for sid in range(N_SENSORS):
    name = f"sensor_{sid:02d}"
    # Base signal: 22°C + small gaussian noise
    temps = 22.0 + np.random.normal(0, 0.3, size=READINGS_PER_SENSOR)

    # Inject DRIFT into sensor_03 starting at hour 80
    if sid == 3:
        drift_start = (80 * 60) // INTERVAL_MIN  # reading index at hour 80
        drift_ramp = np.arange(READINGS_PER_SENSOR - drift_start) * 0.05
        temps[drift_start:] += drift_ramp

    # Inject STUCK-AT into sensor_07 starting at hour 100
    if sid == 7:
        stuck_start = (100 * 60) // INTERVAL_MIN
        temps[stuck_start:] = 21.8

    for i, ts in enumerate(timestamps):
        rows.append((ts, name, round(temps[i], 2)))

df = pd.DataFrame(rows, columns=["timestamp", "sensor_id", "temperature"])

print(f"Total readings: {len(df):,}")
print(f"Sensors: {df['sensor_id'].nunique()}")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()

## Visualize the Raw Data

Let's plot a few sensors to see the faults visually before we try to detect them programmatically.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for ax, sid in zip(axes, ["sensor_00", "sensor_03", "sensor_07"]):
    subset = df[df["sensor_id"] == sid]
    ax.plot(subset["timestamp"], subset["temperature"], linewidth=0.5)
    ax.set_ylabel("°C")
    ax.set_title(sid, loc="left", fontweight="bold")
    ax.axhline(22.0, color="gray", linestyle="--", linewidth=0.5, alpha=0.5)

axes[0].set_title("sensor_00 (healthy)", loc="left", fontweight="bold")
axes[1].set_title("sensor_03 (drifting from hour 80)", loc="left", fontweight="bold")
axes[2].set_title("sensor_07 (stuck from hour 100)", loc="left", fontweight="bold")
plt.tight_layout()
plt.show()

## Detector 1: Rolling Z-Score for Drift

The idea: for each sensor, compute a rolling mean and standard deviation over a trailing window. Then compare each reading against the **fleet-wide** baseline (mean of all healthy sensors at that timestamp). If a sensor's rolling mean deviates more than `Z_THRESHOLD` standard deviations from the fleet baseline, flag it.

This catches gradual drift because the sensor's own rolling mean moves with the drift, but the fleet baseline stays anchored to reality.

In [ ]:
ROLLING_WINDOW = 60  # 60 readings = 5 hours at 5-min intervals
Z_THRESHOLD = 3.0

# Compute fleet baseline: median temperature across all sensors per timestamp
fleet_baseline = df.groupby("timestamp")["temperature"].median().rename("fleet_median")

# Merge baseline back and compute per-sensor rolling stats
df = df.merge(fleet_baseline, on="timestamp")

df["rolling_mean"] = (
    df.groupby("sensor_id")["temperature"]
    .transform(lambda x: x.rolling(ROLLING_WINDOW, min_periods=10).mean())
)

# Z-score: how far is this sensor's rolling mean from the fleet median?
# Use the fleet's rolling std as the denominator
df["fleet_rolling_std"] = (
    df.groupby("timestamp")["temperature"]
    .transform("std")
    .rolling(ROLLING_WINDOW, min_periods=10)
    .mean()
)

df["drift_zscore"] = (df["rolling_mean"] - df["fleet_median"]) / df["fleet_rolling_std"].clip(lower=0.01)
df["drift_flag"] = df["drift_zscore"].abs() > Z_THRESHOLD

# Summary: which sensors got flagged?
flagged_drift = df[df["drift_flag"]].groupby("sensor_id")["timestamp"].agg(["min", "max", "count"])
print("Sensors flagged for DRIFT:")
print(flagged_drift if len(flagged_drift) > 0 else "  (none)")
print(f"\nExpected: sensor_03 flagged starting around hour 80")

## Detector 2: Rolling Variance for Stuck-At Faults

A stuck sensor reports the same (or nearly the same) value every reading. The simplest detector: compute the rolling standard deviation over a window. If it drops to near-zero, the sensor is frozen.

This is cheap to compute and very reliable — a healthy sensor always has *some* noise.

In [ ]:
STUCK_WINDOW = 30       # 30 readings = 2.5 hours
STUCK_STD_THRESHOLD = 0.01  # std below this = sensor is frozen

df["rolling_std"] = (
    df.groupby("sensor_id")["temperature"]
    .transform(lambda x: x.rolling(STUCK_WINDOW, min_periods=10).std())
)

df["stuck_flag"] = df["rolling_std"] < STUCK_STD_THRESHOLD

# Summary
flagged_stuck = df[df["stuck_flag"]].groupby("sensor_id")["timestamp"].agg(["min", "max", "count"])
print("Sensors flagged for STUCK-AT:")
print(flagged_stuck if len(flagged_stuck) > 0 else "  (none)")
print(f"\nExpected: sensor_07 flagged starting around hour 100")

## Visualize Detections

Let's overlay the fault flags on the raw signal to see if the detectors fire at the right time.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# sensor_03: drift detection
s03 = df[df["sensor_id"] == "sensor_03"].copy()
ax = axes[0]
ax.plot(s03["timestamp"], s03["temperature"], linewidth=0.5, label="temperature")
flagged = s03[s03["drift_flag"]]
if len(flagged) > 0:
    ax.axvspan(flagged["timestamp"].min(), flagged["timestamp"].max(),
               alpha=0.2, color="red", label="drift flagged")
ax.set_ylabel("°C")
ax.set_title("sensor_03 — Drift Detection", loc="left", fontweight="bold")
ax.legend(loc="upper left")

# sensor_07: stuck-at detection
s07 = df[df["sensor_id"] == "sensor_07"].copy()
ax = axes[1]
ax.plot(s07["timestamp"], s07["temperature"], linewidth=0.5, label="temperature")
flagged = s07[s07["stuck_flag"]]
if len(flagged) > 0:
    ax.axvspan(flagged["timestamp"].min(), flagged["timestamp"].max(),
               alpha=0.2, color="orange", label="stuck-at flagged")
ax.set_ylabel("°C")
ax.set_title("sensor_07 — Stuck-At Detection", loc="left", fontweight="bold")
ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

## Combined Health Report

In production you'd run both detectors on a schedule and generate a report per sensor. Here's what that looks like for the full fleet.

In [ ]:
report = (
    df.groupby("sensor_id")
    .agg(
        total_readings=("temperature", "count"),
        drift_flags=("drift_flag", "sum"),
        stuck_flags=("stuck_flag", "sum"),
        mean_temp=("temperature", "mean"),
        std_temp=("temperature", "std"),
    )
)

report["drift_pct"] = (report["drift_flags"] / report["total_readings"] * 100).round(1)
report["stuck_pct"] = (report["stuck_flags"] / report["total_readings"] * 100).round(1)
report["status"] = "healthy"
report.loc[report["drift_pct"] > 5, "status"] = "DRIFT"
report.loc[report["stuck_pct"] > 5, "status"] = "STUCK-AT"

print("=== Fleet Health Report ===\n")
print(report[["mean_temp", "std_temp", "drift_pct", "stuck_pct", "status"]].to_string(float_format="%.2f"))

## Takeaways

| Fault Type | Detector | Key Parameter | Catches |
|:---|:---|:---|:---|
| **Drift** | Rolling z-score vs fleet median | `Z_THRESHOLD` (start with 3.0) | Gradual calibration loss, environmental shift on one sensor |
| **Stuck-at** | Rolling std < threshold | `STUCK_STD_THRESHOLD` (start with 0.01) | Frozen readings, dead sensors still reporting last value |

**Tuning tips:**
- **Window size** controls sensitivity vs latency. Shorter windows catch faults faster but produce more false positives from normal variance.
- **Fleet baseline** is only useful when you have multiple co-located sensors. For isolated sensors, compare against a historical baseline instead.
- Neither detector handles **intermittent** faults (sensor flickers between healthy and stuck). For that, look at state-machine approaches or change-point detection (e.g. CUSUM).